# Templated SFT Training Notebook

This notebook is a starting point for someone with a dataset and a base model who wants to run Supervised Fine-Tuning (SFT) on a specific hardware budget. You provide four things — the model, the GPU count, the node count, and the per-GPU VRAM — and the notebook picks memory-influencing hyperparameters that should fit and run end-to-end.

The output is a default configuration you can launch and watch train. It is **not** tuned for best quality or throughput; it is tuned to fit and run. Once it runs, change individual hyperparameters from there.

**Resolves part of [#69](https://github.com/Red-Hat-AI-Innovation-Team/training_hub/issues/69).** Companion notebooks for OSFT and LoRA can follow the same pattern; this one covers SFT only.

## What this notebook does

1. Reads hardware inputs (model, num_gpus, num_nodes, vram_per_gpu) — auto-detects VRAM when possible.
2. Uses `training_hub.estimate` to walk a list of candidate `max_tokens_per_gpu` values and pick the largest one whose expected memory fits within a safety margin of the declared VRAM.
3. Sets sensible defaults for the remaining training params (epochs, learning rate, effective batch size, max_seq_len).
4. Calls `training_hub.sft` with the resulting configuration.

## What this notebook does *not* do

- Tune for downstream task quality. Loss-curve-and-eval iteration is on you after the first run.
- Validate that your dataset is well-formed. Use the dataset checks from the comprehensive tutorial if you're not sure.
- Optimize aggressively. The candidate list and safety margin are conservative on purpose — the goal is "runs out of the box," not "squeezes every last token onto the GPU."

## Setup

In [ ]:
import torch
from training_hub import sft, estimate

## Step 1: Hardware inputs

Set these four values for your environment. Everything below is derived from them.

- **`model_path`** — HuggingFace model ID or local path to the base model.
- **`num_gpus`** — GPUs per node.
- **`num_nodes`** — Number of nodes (1 for a single-machine run).
- **`vram_per_gpu_gb`** — Per-GPU VRAM in GB. Leave as `None` to auto-detect via CUDA. Common values: H100/A100 80GB, A100 40GB, L40/L40S/A6000 48GB, RTX 4090/3090 24GB, RTX 4080 16GB.

In [ ]:
# === HARDWARE INPUTS ===
model_path = "Qwen/Qwen2.5-1.5B-Instruct"
num_gpus = 1
num_nodes = 1
vram_per_gpu_gb = None  # None = auto-detect from CUDA

In [ ]:
# Validate inputs before doing anything expensive.
# Note: bool is a subclass of int in Python, so True/False would otherwise pass
# isinstance(x, int) silently — reject explicitly.
if not isinstance(num_gpus, int) or isinstance(num_gpus, bool) or num_gpus < 1:
    raise ValueError("num_gpus must be a positive integer.")
if not isinstance(num_nodes, int) or isinstance(num_nodes, bool) or num_nodes < 1:
    raise ValueError("num_nodes must be a positive integer.")
if vram_per_gpu_gb is not None and (
    isinstance(vram_per_gpu_gb, bool)
    or not isinstance(vram_per_gpu_gb, (int, float))
    or vram_per_gpu_gb <= 0
):
    raise ValueError("vram_per_gpu_gb must be > 0 when provided, or None to auto-detect.")

# Auto-detect VRAM if not set, otherwise trust the user's value.
if vram_per_gpu_gb is None:
    if not torch.cuda.is_available():
        raise RuntimeError(
            "vram_per_gpu_gb is None but CUDA is not available. "
            "Set vram_per_gpu_gb explicitly to the per-GPU VRAM in GB."
        )
    vram_per_gpu_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    # When multiple GPUs are visible, auto-correct to the smallest GPU's memory
    # so the sizing is safe across the whole set; sizing for GPU 0 would OOM
    # smaller siblings.
    if torch.cuda.device_count() > 1:
        gpu0_mem = torch.cuda.get_device_properties(0).total_memory
        other_mems = [torch.cuda.get_device_properties(i).total_memory for i in range(1, torch.cuda.device_count())]
        if any(m != gpu0_mem for m in other_mems):
            min_gb = min([gpu0_mem] + other_mems) / (1024 ** 3)
            vram_per_gpu_gb = min_gb
            print(
                f"WARNING: {torch.cuda.device_count()} GPUs with non-uniform memory detected. "
                f"Auto-corrected to smallest GPU's VRAM ({min_gb:.0f} GB). "
                f"Set vram_per_gpu_gb explicitly to override."
            )
    print(f"Using {vram_per_gpu_gb:.0f} GB VRAM per GPU ({torch.cuda.get_device_name(0)}{', ' + str(torch.cuda.device_count() - 1) + ' more' if torch.cuda.device_count() > 1 else ''}).")

vram_per_gpu_bytes = int(vram_per_gpu_gb * (1024 ** 3))
total_gpus = num_gpus * num_nodes
print(f"Training plan: {total_gpus} GPU(s) across {num_nodes} node(s), {vram_per_gpu_gb:.0f} GB each.")

## Step 2: Pick `max_tokens_per_gpu` that fits

SFT memory usage scales roughly with `max_tokens_per_gpu` (the cap on the number of tokens placed on a single GPU per minibatch). Throughput scales with it too, so we want the largest value that still fits.

The cell below walks a list of candidate values (largest first), asks `training_hub.estimate` for the expected memory usage at each, and picks the first one whose expected usage is within `safety_margin` of the declared VRAM. If even the smallest candidate doesn't fit, it raises with a list of options (smaller model, larger GPUs, LoRA instead of full SFT).

Adjust `candidates` or `safety_margin` if you have a specific tradeoff in mind.

In [ ]:
candidates = [16384, 8192, 4096, 2048, 1024, 512, 256]
safety_margin = 0.90  # use no more than 90% of declared VRAM

max_tokens_per_gpu = None
selected_expected_bytes = None

for candidate in candidates:
    try:
        _, expected_bytes, _ = estimate(
            training_method="sft",
            num_gpus=total_gpus,
            gpu_memory=vram_per_gpu_bytes,
            model_path=model_path,
            max_tokens_per_gpu=candidate,
            verbose=0,
        )
    except Exception as e:
        print(f"  estimate() failed at candidate={candidate}: {type(e).__name__}: {e}. Trying smaller.")
        continue
    if expected_bytes is None or expected_bytes <= 0:
        print(f"  estimate() returned non-positive value at candidate={candidate}. Trying smaller.")
        continue
    if expected_bytes <= vram_per_gpu_bytes * safety_margin:
        max_tokens_per_gpu = candidate
        selected_expected_bytes = expected_bytes
        break

if max_tokens_per_gpu is None:
    raise RuntimeError(
        f"None of {candidates} fits in {vram_per_gpu_gb:.0f} GB per GPU for model {model_path}.\n"
        "Options to try:\n"
        "  - Use a smaller base model\n"
        "  - Add more or larger GPUs\n"
        "  - Switch to LoRA (see the LoRA templated notebook when available)\n"
        "  - Raise safety_margin (only if you accept higher OOM risk)"
    )

pct_used = selected_expected_bytes / vram_per_gpu_bytes * 100
print(
    f"Selected max_tokens_per_gpu = {max_tokens_per_gpu} "
    f"(expected memory: {selected_expected_bytes / (1024 ** 3):.1f} GB per GPU, {pct_used:.0f}% of VRAM)."
)

## Step 3: Data and training config

Set the data path, output directory, and the rest of the hyperparameters. The defaults below are conservative — they aim for a stable first run, not a tuned one.

- **`data_path`** — JSONL file in `messages` format (see the SFT docs).
- **`ckpt_output_dir`** — Where checkpoints land.
- **`max_seq_len`** — Capped at `max_tokens_per_gpu` (training_hub will error if larger).
- **`effective_batch_size`** — Defaulted to one sample per GPU; raise via gradient accumulation if your data and memory allow.
- **`learning_rate`** — `2e-5` is a sensible SFT default for most decoder-only LLMs; tune after you see the loss curve.

In [ ]:
# === DATA AND OUTPUT ===
data_path = "/path/to/your/training_data.jsonl"
ckpt_output_dir = "/path/to/checkpoint/output"

# === TRAINING HYPERPARAMETERS (tune from here) ===
num_epochs = 3
effective_batch_size = max(1, total_gpus)
learning_rate = 2e-5
max_seq_len = min(4096, max_tokens_per_gpu)

print(f"num_epochs:           {num_epochs}")
print(f"effective_batch_size: {effective_batch_size}")
print(f"learning_rate:        {learning_rate}")
print(f"max_seq_len:          {max_seq_len}  (must be <= max_tokens_per_gpu = {max_tokens_per_gpu})")

## Step 4: Launch SFT

Single-node and multi-node both work. For multi-node, set `rdzv_endpoint`, `node_rank`, `master_addr`, `master_port`, etc. — see `src/training_hub/algorithms/sft.py` for the full parameter list.

Update `data_path` and `ckpt_output_dir` above to real paths before running this cell.

In [ ]:
result = sft(
    model_path=model_path,
    data_path=data_path,
    ckpt_output_dir=ckpt_output_dir,
    num_epochs=num_epochs,
    effective_batch_size=effective_batch_size,
    learning_rate=learning_rate,
    max_seq_len=max_seq_len,
    max_tokens_per_gpu=max_tokens_per_gpu,
    nproc_per_node=num_gpus,
    nnodes=num_nodes,
)
print(result)

## Where to go from here

- **First, confirm it ran.** A clean SFT run on a small model usually finishes a few hundred steps in tens of minutes. If you hit OOM despite the safety margin, lower `max_tokens_per_gpu` or lower `safety_margin` (more conservative), or pick the next-smaller candidate manually.
- **Then look at the loss curve.** `training_hub.plot_loss` reads the JSONL metrics emitted to `ckpt_output_dir` and renders it. Diverging or flat curves usually mean `learning_rate` is wrong, not the memory config.
- **Then tune for your task.** `effective_batch_size` (gradient accumulation), `learning_rate`, and `num_epochs` are the highest-leverage knobs once the run is stable. The dataset shape matters more than any of these.
- **For comprehensive guidance** (logging, validation, multi-node, pretraining mode, etc.) see `sft_comprehensive_tutorial.ipynb`.